In [ ]:
# Code used to visualize Grad-CAM heatmaps on model layers

from torch.utils.data import DataLoader
import numpy as np
import os
import torch
from utils import dataset_precip
from tqdm import tqdm
from models import SmaAT_UNet_VQ_lightning
import matplotlib.pyplot as plt
import warnings
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam import GradCAM
from argparse import Namespace

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')


class SemanticSegmentationTarget:
    def __init__(self, category, mask, device):
        self.category = category
        self.mask = torch.from_numpy(mask)
        if device == 'cuda':
            self.mask = self.mask.cuda()

    def __call__(self, model_output):
        return (model_output[self.category, :, :] * self.mask).sum()


# def load_model(model_cls, model_folder, device):
#     models = [m for m in os.listdir(model_folder) if ".ckpt" in m]
#     model_file = models[-1]
#     model = model_cls.load_from_checkpoint(f"{model_folder}/{model_file}")

#     model.hparams["model_type"] = "partialmixconv"

#     model.eval()
#     model.to(torch.device(device))
#     return model

def load_model(model_cls, model_folder, device, model_type="partialmixconv"):
    # 1) Pick the latest .ckpt in model_folder
    ckpts = [fn for fn in os.listdir(model_folder) if fn.endswith(".ckpt")]
    if not ckpts:
        raise FileNotFoundError(f"No .ckpt found in {model_folder!r}")
    ckpt_file = sorted(ckpts)[-1]
    ckpt_path = os.path.join(model_folder, ckpt_file)

    # 2) Load the checkpoint dict
    checkpoint = torch.load(ckpt_path, map_location=device)

    # 3) Extract and possibly modify hyperparameters
    hparams_dict = checkpoint.get("hyper_parameters", {})
    if "model_type" not in hparams_dict:
        hparams_dict["model_type"] = model_type

    # 4) Wrap into a single Namespace so __init__(self, hparams) sees everything
    hparams_ns = Namespace(**hparams_dict)

    # 5) Instantiate the model with that Namespace
    model = model_cls(hparams_ns)

    # 6) Load weights (strip any "model." prefix that Lightning may have added)
    state_dict = checkpoint.get("state_dict", checkpoint)
    new_state_dict = {}
    for key, val in state_dict.items():
        if key.startswith("model."):
            new_key = key[len("model."):]
        else:
            new_key = key
        new_state_dict[new_key] = val

    model.load_state_dict(new_state_dict)

    # 7) Switch to eval and move to the correct device
    model.eval()
    model.to(torch.device(device))
    return model

def get_segmentation_data(in_channels):
    dataset = dataset_precip.precipitation_maps_oversampled_h5(
        in_file=data_file,
        num_input_images=in_channels,
        num_output_images=6,
        train=False)

    test_dl = torch.utils.data.DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )
    return test_dl


def run_cam(model, target_layers, device, in_channels):
    test_dl = get_segmentation_data(in_channels)
    for x, y_true in tqdm(test_dl, leave=False):
        x = x.to(torch.device(device))

        # We only care about logits
        logits, vq_loss, loss_dict = model(x)

        mask = np.digitize((logits[0][0] * 47.83 * 12).detach().cpu().numpy(), np.array([1.5]), right=True)
        mask_float = np.float32(mask)
        print("Total mask pixels:", mask_float.sum())

        image = torch.stack([x[0][0], x[0][0], x[0][0]], dim=2)
        image = image.cpu().numpy()
        targets = [SemanticSegmentationTarget(0, mask_float, device)]
        cam_image = []
        for layer in target_layers:
            with GradCAM(model=model, target_layers=layer) as cam:
                grayscale_cam = cam(input_tensor=x, targets=targets)[0, :]
                cam_image.append(show_cam_on_image(image, grayscale_cam, use_rgb=True))

        # fig, axes = plt.subplots(2, 3, figsize=(16, 16))
        # axes[0][0].imshow(cam_image[0])
        # axes[0][0].set_title('Up1 Activation')
        # axes[0][1].imshow(cam_image[1])
        # axes[0][1].set_title('Up2 Activation')
        # axes[1][0].imshow(cam_image[2])
        # axes[1][0].set_title('Up3 Activation')
        # axes[1][1].imshow(cam_image[3])
        # axes[1][1].set_title('Up4 Activation')
        # axes[1][2].imshow(cam_image[4])
        # axes[1][2].set_title("OutC Activation")

        # plt.show()

        fig, axes = plt.subplots(3, 4, figsize=(24, 18))
        titles = [
            'Inc', 'Down1', 'Down2', 'Down3',
            'Down4', 'CBAM5',
            'Up1', 'Up2', 'Up3', 'Up4', 'OutC'
        ]
        for idx, (ax_row, ax_col) in enumerate([(i,j) for i in range(3) for j in range(4)][:len(cam_image)]):
            axes[ax_row][ax_col].imshow(cam_image[idx])
            axes[ax_row][ax_col].set_title(titles[idx], fontsize=14)
            axes[ax_row][ax_col].axis('off')

        plt.tight_layout()
        plt.show()

        # break


if __name__ == '__main__':
    hparams = {
        'model': 'SmaAT_UNet_VQ_MSE_PartialMixConv',
        'out_channels': 1,
        "n_channels": 12,
        "n_classes": 1,
        "reduction_ratio": 16,
        "kernels_per_layer": 1,
        "batch_size": 8,
        "learning_rate": 0.001,
        'gpus': -1,
        "lr_patience": 4,
        "es_patience": 15,
        "use_oversampled_dataset": True,
        "bilinear": True,
        "valid_size": 0.1,
        "dataset_folder": "data/precipitation/train_test_2016-2019_input-length_12_img-ahead_6_rain-threshold_50.h5",
        # change input-length and img-ahead accordingly
        "resume_from_checkpoint": None
        # f"{args.model}/ResSmaAt_UNet2_rain_threshold_50_epoch=56-val_loss=0.300085.ckpt"
    }

    SmaAT_UNet_VQ = SmaAT_UNet_VQ_lightning.SmaAT_UNet_VQ

    model_folder = "checkpoints/comparisongradcam"
    data_file = 'data/precipitation/train_test_2016-2019_input-length_12_img-ahead_6_rain-threshold_50.h5'
    device = 'cpu'
    model = load_model(SmaAT_UNet_VQ, model_folder, device, model_type="partialmixconv")
    print(model)
    target_layers = [
        [model.down1.maxpool_conv[1].double_conv[0]],
        [model.down2.maxpool_conv[1].double_conv[0]],
        [model.down3.maxpool_conv[1].mix1[0]],
        [model.down4.maxpool_conv[1].mix1[0]],
        [model.cbam5],                           
        [model.up1.conv.mix1[0]],
        [model.up2.conv.double_conv[0]],
        [model.up3.conv.double_conv[0]],
        [model.up4.conv.double_conv[0]],
        [model.outc.conv],       
    ]
    run_cam(model, target_layers, device, hparams['n_channels'])